In [22]:
from pathlib import Path
import yaml
from skimage.measure import label, regionprops
import cv2
from tqdm.notebook import tqdm
import pandas as pd
from ultralytics import YOLO
from typing import Tuple

In [23]:
TEST_CSV_PATH = Path("./data/DeepFish/Segmentation/test.csv")
TRAIN_CSV_PATH = Path("./data/DeepFish/Segmentation/train.csv")
VAL_CSV_PATH = Path("./data/DeepFish/Segmentation/val.csv")

TEST_CSV_PATH.exists(), TRAIN_CSV_PATH.exists(), VAL_CSV_PATH.exists()

(True, True, True)

In [24]:
test_df = pd.read_csv(TEST_CSV_PATH)
train_df = pd.read_csv(TRAIN_CSV_PATH)
val_df = pd.read_csv(VAL_CSV_PATH)

In [25]:
IMAGE_DIR = Path("../datasets/DeepFish/Detection/images")
IMAGE_TRAIN_DIR = IMAGE_DIR / "train"
IMAGE_VAL_DIR = IMAGE_DIR / "val"
IMAGE_TEST_DIR = IMAGE_DIR / "test"

IMAGE_DIR.mkdir(parents=True, exist_ok=True)
IMAGE_TRAIN_DIR.mkdir(parents=True, exist_ok=True)
IMAGE_VAL_DIR.mkdir(parents=True, exist_ok=True)
IMAGE_TEST_DIR.mkdir(parents=True, exist_ok=True)

LABEL_DIR = Path("../datasets/DeepFish/Detection/labels")
LABEL_TRAIN_DIR = LABEL_DIR / "train"
LABEL_VAL_DIR = LABEL_DIR / "val"
LABEL_TEST_DIR = LABEL_DIR / "test"

LABEL_DIR.mkdir(parents=True, exist_ok=True)
LABEL_TRAIN_DIR.mkdir(parents=True, exist_ok=True)
LABEL_VAL_DIR.mkdir(parents=True, exist_ok=True)
LABEL_TEST_DIR.mkdir(parents=True, exist_ok=True)

In [26]:
SEGMENTATION_IMAGES = list(Path("./data/DeepFish/Segmentation/images").glob("*/*.jpg"))
SEGMENTATION_MASKS = list(Path("./data/DeepFish/Segmentation/masks").glob("*/*.png"))

len(SEGMENTATION_IMAGES), len(SEGMENTATION_MASKS)

(620, 620)

In [27]:
segmentation_mask_map = {mask.stem: mask for mask in SEGMENTATION_MASKS}
segmentation_image_mask_pairs = [(image, segmentation_mask_map[image.stem]) for image in SEGMENTATION_IMAGES if image.stem in segmentation_mask_map]

len(segmentation_image_mask_pairs)

620

In [28]:
def segmentation_to_bounding_boxes(segmentation_mask: Path) -> list[tuple[int, int, int, int]]:
    mask = cv2.imread(str(segmentation_mask), cv2.IMREAD_UNCHANGED)
    labeled_mask = label(mask)
    regions = regionprops(labeled_mask)
    bounding_boxes = []
    for region in regions:
        # region.bbox -> (min_row, min_col, max_row, max_col)
        min_row, min_col, max_row, max_col = region.bbox
        # convert to (x_min, y_min, x_max, y_max) in pixel coords
        x_min, y_min, x_max, y_max = min_col, min_row, max_col, max_row
        bounding_boxes.append((x_min, y_min, x_max, y_max))
    return bounding_boxes

In [29]:
image_bounding_box_pairs = [(image, segmentation_to_bounding_boxes(mask)) for image, mask in segmentation_image_mask_pairs]

In [ ]:
# point to the actual dataset location (datasets was moved up one level)
deepfish_root = Path("../datasets/DeepFish").resolve()
deepfish_yaml = {
    "path": str(deepfish_root),
    "train": "Detection/images/train",
    "val": "Detection/images/val",
    "test": "Detection/images/test",
    "names": ["fish"],
}

# write YAML at repo root so ultralytics resolves absolute paths reliably
with open("./deepfish_detect.yaml", "w") as f:
    yaml.dump(deepfish_yaml, f)

In [31]:
def get_dir_for_image(image: Path) -> Tuple[Path, Path]:
    image_id = image.stem
    if image_id in {value.split("/")[-1] for value in test_df["ID"].values}:
        return IMAGE_TEST_DIR, LABEL_TEST_DIR
    elif image_id in {value.split("/")[-1] for value in train_df["ID"].values}:
        return IMAGE_TRAIN_DIR, LABEL_TRAIN_DIR
    elif image_id in {value.split("/")[-1] for value in val_df["ID"].values}:
        return IMAGE_VAL_DIR, LABEL_VAL_DIR
    else:
        raise ValueError(f"Image {image} not found in any of the CSV files.")

In [35]:
for image, bounding_boxes in tqdm(image_bounding_box_pairs):
    image_name = image.name
    image_path, label_path = get_dir_for_image(image)
    image_path = image_path  / image_name
    image_path.parent.mkdir(parents=True, exist_ok=True)
    if not image_path.exists():
        image_path.symlink_to(image.resolve())

    image_array = cv2.imread(str(image_path))
    img_height, img_width = image_array.shape[:2]

    label_path = label_path / f"{image.stem}.txt"
    with open(label_path, "w") as f:
        for bbox in bounding_boxes:
            x_min, y_min, x_max, y_max = bbox
            x_center = (x_min + x_max) / 2.0
            y_center = (y_min + y_max) / 2.0
            width = float(x_max - x_min)
            height = float(y_max - y_min)

            # Normalize the coordinates to [0, 1]
            x_center /= float(img_width)
            y_center /= float(img_height)
            width /= float(img_width)
            height /= float(img_height)

            # write with fixed precision to avoid scientific notation or extra long floats
            f.write(f"0 {x_center:.6f} {y_center:.6f} {width:.6f} {height:.6f}\n")

  0%|          | 0/620 [00:00<?, ?it/s]

In [37]:
model = YOLO("yolo26n.pt")

In [ ]:
results = model.train(data="./deepfish_detect.yaml", epochs=100, imgsz=640)

Ultralytics 8.4.19 🚀 Python-3.13.9 torch-2.10.0+cu128 CUDA:0 (NVIDIA GeForce RTX 3060 Laptop GPU, 5804MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=./deepfish.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, 